<a href="https://colab.research.google.com/github/prathamesh-kandpal/FERIA/blob/main/FERIA_Code_Notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# FERIA
### Financial Email Resolution & Inquiry Agent

**Team:** Nexus AI

**Team Members:** Prathamesh, Prajay, Srishti, Mansi, Shefali, Gianna, Priyanshi, Priyanka

## Objective
Build an agentic AI system that:

1. Accepts customer emails
2. Classifies customer intent
3. Redacts sensitive information
4. Retrieves relevant policy context using RAG
5. Drafts compliant responses
6. Escalates high-risk cases to human agents



# System Architecture

## High-Level Flow

Customer Email
   
↓

Triage Agent

↓

PII Redaction Agent

↓

RAG Retrieval Agent

↓

Response Drafting Agent

↓

Final Response / Escalation

---

## Components

### Backend
- FastAPI
- CrewAI

### RAG
- LangChain
- FAISS / ChromaDB

### LLM
- Claude / OpenAI

### Frontend
- HTML
- CSS
- JavaScript

###1. Core Imports

In [ ]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Optional, Dict, Any, List
from pydantic import BaseModel, Field

# LLM (placeholder - configure as needed)
# from langchain.chat_models import ChatOpenAI / Claude wrapper

###2. Define Graph State

In [ ]:
class FERIAState(TypedDict):
    """
    Shared state flowing through the FERIA pipeline.

    This state captures the full lifecycle of an insurance support email,
    from ingestion to resolution or escalation.
    """

    # Raw + processed input
    raw_email: str
    customer_name: Optional[str]
    email_subject: Optional[str]
    redacted_email: Optional[str]

    # PII tracking
    pii_entities: Optional[List[str]]

    # Triage outputs
    insurance_line: Optional[str]
    query_type: Optional[str]
    urgency: Optional[str]
    risk_flags: Optional[List[str]]
    escalation_required: Optional[bool]
    sla_tag: Optional[str]

    # Retrieval outputs
    retrieved_documents: Optional[List[Dict[str, Any]]]

    # Resolution
    draft_response: Optional[str]
    confidence_score: Optional[float]

    # Escalation
    escalation_context: Optional[Dict[str, Any]]

    # Audit
    audit_log: Optional[Dict[str, Any]]

### 3. Define Structured Schemas (Pydantic)

Triage Output

In [ ]:
class TriageOutput(BaseModel):
    """
    Structured classification output for the triage agent.
    """

    insurance_line: str = Field(..., description="Health, Motor, Life, Travel, Home")
    query_type: str = Field(..., description="Type of customer query")
    urgency: str = Field(..., description="Critical, High, Medium, Low")
    risk_flags: List[str] = Field(default_factory=list)
    escalation_required: bool
    sla_tag: str

Retrieval Output

In [ ]:
class RetrievalOutput(BaseModel):
    """
    Output schema for retrieved policy documents.
    """

    documents: List[Dict[str, Any]] = Field(
        ..., description="Relevant policy clauses with metadata"
    )

Resolution Output

In [ ]:
class ResolutionOutput(BaseModel):
    """
    Output schema for generated response.
    """

    response: str
    confidence_score: float

###4. Initialize LLM

In [ ]:
# Placeholder LLM initialization
# Replace with Claude / GPT / other model

llm = None

###5. Build Nodes

5.1 Email Ingestion Node

In [ ]:
def email_ingestion_node(state: FERIAState) -> FERIAState:
    """
    Ingests raw email input into the pipeline.

    Expected Responsibilities:
    - Accept incoming email text
    - Normalize formatting if required
    - Initialize audit trail entry
    """
    pass

5.2 PII Redaction Node

In [ ]:
def pii_redaction_node(state: FERIAState) -> FERIAState:
    """
    Redacts sensitive information from the email.

    Expected Responsibilities:
    - Detect PII (policy numbers, Aadhaar, PAN, etc.)
    - Replace with placeholders
    - Store redacted entity types in audit log
    - Output redacted_email
    """
    pass

5.3 Triage Node

In [ ]:
def triage_node(state: FERIAState) -> FERIAState:
    """
    Classifies the email into structured triage categories.

    Expected Responsibilities:
    - Identify insurance line
    - Detect query type
    - Assign urgency
    - Detect risk flags
    - Decide escalation requirement
    - Assign SLA tag
    """
    pass

5.4 Retrieval (RAG) Node

In [ ]:
def retrieval_node(state: FERIAState) -> FERIAState:
    """
    Retrieves relevant policy documents based on triage output.

    Expected Responsibilities:
    - Query vector database using insurance_line + query_type
    - Retrieve relevant clauses
    - Attach metadata (policy name, section, clause)
    """
    pass

5.5 Resolution Node

In [ ]:
def resolution_node(state: FERIAState) -> FERIAState:
    """
    Generates a customer-facing response for non-escalated cases.

    Expected Responsibilities:
    - Use retrieved documents as grounding
    - Generate clear, empathetic response
    - Include citations
    - Assign confidence score
    """
    pass

5.6 Escalation Handler Node

In [ ]:
def escalation_node(state: FERIAState) -> FERIAState:
    """
    Prepares escalation context for human-in-the-loop handling.

    Expected Responsibilities:
    - Compile full case summary
    - Attach triage outputs
    - Attach retrieved documents
    - Include draft response (if any)
    - Prepare assistant-ready context
    """
    pass

5.7 Audit Node

In [ ]:
def audit_node(state: FERIAState) -> FERIAState:
    """
    Finalizes and stores audit trail for the case.

    Expected Responsibilities:
    - Log PII types redacted
    - Store triage classification
    - Store retrieved document references
    - Record escalation decision
    - Capture SLA + timestamps
    """
    pass

###6. Routing Function

In [ ]:
def route_after_triage(state: FERIAState) -> str:
    """
    Determines whether to proceed to auto-resolution or escalation.

    Routing Logic:
    - If escalation_required is True → escalation_node
    - Else → retrieval_node
    """
    if state.get("escalation_required"):
        return "escalation_node"
    return "retrieval_node"

Post-Resolution Routing

In [ ]:
def route_after_resolution(state: FERIAState) -> str:
    """
    Determines if low-confidence responses should be escalated.

    Routing Logic:
    - If confidence_score is below threshold → escalation_node
    - Else → audit_node
    """
    confidence = state.get("confidence_score", 0)

    if confidence < 0.7:
        return "escalation_node"

    return "audit_node"

###7. Build & Compile Graph

In [ ]:
def build_feria_graph():
    """
    Constructs and compiles the FERIA LangGraph workflow.

    Flow:
    START
      → ingestion
      → pii_redaction
      → triage
      → (conditional)
          → retrieval → resolution → (conditional) → audit / escalation
          → escalation
      → audit
      → END
    """

    builder = StateGraph(FERIAState)

    # Add nodes
    builder.add_node("email_ingestion", email_ingestion_node)
    builder.add_node("pii_redaction", pii_redaction_node)
    builder.add_node("triage", triage_node)
    builder.add_node("retrieval_node", retrieval_node)
    builder.add_node("resolution_node", resolution_node)
    builder.add_node("escalation_node", escalation_node)
    builder.add_node("audit_node", audit_node)

    # Linear flow
    builder.add_edge(START, "email_ingestion")
    builder.add_edge("email_ingestion", "pii_redaction")
    builder.add_edge("pii_redaction", "triage")

    # Conditional routing after triage
    builder.add_conditional_edges(
        "triage",
        route_after_triage,
        {
            "retrieval_node": "retrieval_node",
            "escalation_node": "escalation_node",
        },
    )

    # Retrieval → Resolution
    builder.add_edge("retrieval_node", "resolution_node")

    # Conditional after resolution
    builder.add_conditional_edges(
        "resolution_node",
        route_after_resolution,
        {
            "audit_node": "audit_node",
            "escalation_node": "escalation_node",
        },
    )

    # Escalation → Audit
    builder.add_edge("escalation_node", "audit_node")

    # End
    builder.add_edge("audit_node", END)

    return builder.compile()

###8. Visualize the Graph

In [ ]:
from IPython.display import Image, display

graph = build_feria_graph()

display(Image(graph.get_graph().draw_mermaid_png()))

# FERIA — Financial Email Resolution & Inquiry Agent
### by Nexus AI

> Agentic AI pipeline for insurance customer email support | Google Colab Notebook

---

## Notebook Structure

| Section | Description | Owner |
|---------|-------------|-------|
| 0 | Environment Setup & Installs | All |
| 1 | Folder Structure & Config | [Name] |
| 2 | Data Loading — Policy Docs & Emails | [Name] |
| 3 | PII Redaction Module | [Name] |
| 4 | RAG Pipeline — Embeddings & ChromaDB | [Name] |
| 5 | LangChain Agents — Triage, RAG, Resolution | [Name] |
| 6 | Escalation Handler Bot | [Name] |
| 7 | Audit Trail & SQLite Logging | [Name] |
| 8 | FastAPI Backend + Static HTML Server | [Name] |

---
> **Ground rules:** Do not modify another person's section without checking with them first.
> Each section is self-contained. Run them top to bottom.


---
## Section 0 — Environment Setup & Installs
**Owner: All team members**

Run this once at the start of every Colab session.


In [ ]:
# ── Section 0: Install dependencies ─────────────────────────────────────────
# Run this cell first, every session

!pip install -q langchain langchain-community langchain-anthropic
!pip install -q chromadb
!pip install -q pymupdf          # PyMuPDF for PDF parsing
!pip install -q spacy
!python -m spacy download en_core_web_sm -q
!pip install -q fastapi uvicorn[standard] python-dotenv pydantic
!pip install -q sqlite-utils

print("All dependencies installed.")


In [ ]:
# ── Section 0: API Keys & Environment ───────────────────────────────────────
# Option A: Set directly (do NOT commit with real keys)
# Option B: Use Colab Secrets (recommended) — add via the key icon in the sidebar

import os

# Uncomment and fill in if not using Colab Secrets
# os.environ["ANTHROPIC_API_KEY"] = "your-key-here"

# If using Colab Secrets:
from google.colab import userdata
os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")

print("Environment ready.")


In [ ]:
# ── Section 0: API Keys & Environment ───────────────────────────────────────
# Option A: Set directly (do NOT commit with real keys)
# Option B: Use Colab Secrets (recommended) — add via the key icon in the sidebar

import os

# Uncomment and fill in if not using Colab Secrets
# os.environ["ANTHROPIC_API_KEY"] = "your-key-here"

# If using Colab Secrets:
from google.colab import userdata
os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")

print("Environment ready.")


---
## Section 1 — Folder Structure & Config
**Owner: [Name]**

Sets up the project folder layout and a shared config object used across all sections.
Run once per session after mounting Drive.


In [ ]:
# ── Section 1: Mount Google Drive ────────────────────────────────────────────
from google.colab import drive
drive.mount("/content/drive")

# Set this to your shared Drive folder path
PROJECT_ROOT = "/content/drive/MyDrive/FERIA"
print(f"Project root: {PROJECT_ROOT}")


In [ ]:
# ── Section 1: Create folder structure ───────────────────────────────────────
import os

FOLDERS = [
    "data/policy_docs/health",
    "data/policy_docs/motor",
    "data/policy_docs/life",
    "data/policy_docs/travel",
    "data/policy_docs/home",
    "data/emails",
    "data/sop",
    "vectordb",
    "db",
    "static/css",
    "static/js",
    "logs",
]

for folder in FOLDERS:
    path = os.path.join(PROJECT_ROOT, folder)
    os.makedirs(path, exist_ok=True)

print("Folder structure ready:")
for f in FOLDERS:
    print(f"  {PROJECT_ROOT}/{f}/")


In [ ]:
# ── Section 1: Shared Config ─────────────────────────────────────────────────
# All sections import from this config dict — do not duplicate paths elsewhere

CONFIG = {
    # Paths
    "project_root":      PROJECT_ROOT,
    "policy_docs_dir":   f"{PROJECT_ROOT}/data/policy_docs",
    "emails_dir":        f"{PROJECT_ROOT}/data/emails",
    "sop_dir":           f"{PROJECT_ROOT}/data/sop",
    "vectordb_dir":      f"{PROJECT_ROOT}/vectordb",
    "db_path":           f"{PROJECT_ROOT}/db/feria.db",
    "static_dir":        f"{PROJECT_ROOT}/static",
    "logs_dir":          f"{PROJECT_ROOT}/logs",

    # Insurance lines — used for routing across all agents
    "insurance_lines":   ["health", "motor", "life", "travel", "home"],

    # RAG
    "chunk_size":        800,
    "chunk_overlap":     100,
    "top_k":             4,

    # LLM
    "llm_model":         "claude-sonnet-4-6",
    "llm_temperature":   0.2,

    # App
    "app_version":       "1.0.0",
    "auto_resolve_confidence_threshold": 0.70,
}

print("Config loaded.")
for k, v in CONFIG.items():
    print(f"  {k}: {v}")


---
## Section 2 — Data Loading: Policy Documents & Emails
**Owner: [Name]**

Loads policy PDFs from `data/policy_docs/` and raw customer emails from `data/emails/`.
Outputs clean text objects passed into Section 3 (PII Redaction) and Section 4 (RAG).

**Expected folder layout:**
```
data/
├── policy_docs/
│   ├── health/    → health_policy.pdf, ...
│   ├── motor/     → motor_policy.pdf, ...
│   ├── life/      → life_policy.pdf, ...
│   ├── travel/    → travel_policy.pdf, ...
│   └── home/      → home_policy.pdf, ...
├── emails/        → customer_email_001.txt, ...
└── sop/           → irdai_guidelines.pdf, escalation_sop.txt, ...
```


In [ ]:
# ── Section 2: Load Policy PDFs using PyMuPDF ────────────────────────────────
# Returns: policy_docs — list of dicts with keys: text, source, insurance_line

import fitz  # PyMuPDF
import os

def load_policy_pdfs(policy_docs_dir: str, insurance_lines: list) -> list[dict]:
    """
    Walk each insurance line subfolder.
    Parse every PDF found and return a list of document dicts.
    Each dict: { "text": str, "source": filename, "insurance_line": str }
    """
    docs = []

    # TODO: iterate over insurance_lines
    # TODO: for each line, walk the subfolder for .pdf files
    # TODO: open each PDF with fitz.open(filepath)
    # TODO: extract text from all pages: page.get_text()
    # TODO: append { "text": full_text, "source": filename, "insurance_line": line }

    print(f"Loaded {len(docs)} policy documents.")
    return docs

policy_docs = load_policy_pdfs(CONFIG["policy_docs_dir"], CONFIG["insurance_lines"])


In [ ]:
# ── Section 2: Load SOP and Guideline Documents ──────────────────────────────
# Returns: sop_docs — list of dicts (same schema as policy_docs)
# Handles both .pdf and .txt files

def load_sop_docs(sop_dir: str) -> list[dict]:
    """
    Load IRDAI guidelines, internal SOPs, and FAQ documents.
    Supports .pdf (via PyMuPDF) and .txt files.
    Each dict: { "text": str, "source": filename, "insurance_line": "general" }
    """
    docs = []

    # TODO: walk sop_dir for .pdf and .txt files
    # TODO: for .pdf — use fitz.open(), extract all page text
    # TODO: for .txt — open with utf-8 encoding, read full content
    # TODO: append { "text": ..., "source": filename, "insurance_line": "general" }

    print(f"Loaded {len(docs)} SOP/guideline documents.")
    return docs

sop_docs = load_sop_docs(CONFIG["sop_dir"])
all_docs = policy_docs + sop_docs
print(f"Total documents loaded: {len(all_docs)}")


In [ ]:
# ── Section 2: Load Customer Emails ──────────────────────────────────────────
# Returns: raw_emails — list of dicts
# Each .txt file in data/emails/ is one customer email

def load_emails(emails_dir: str) -> list[dict]:
    """
    Load all .txt email files from the emails directory.
    Each dict: { "id": str, "filename": str, "raw_text": str }
    Email ID is the filename without extension.
    """
    emails = []

    # TODO: walk emails_dir for .txt files
    # TODO: read each file with utf-8 encoding
    # TODO: append { "id": stem, "filename": fname, "raw_text": content }

    print(f"Loaded {len(emails)} customer emails.")
    return emails

raw_emails = load_emails(CONFIG["emails_dir"])

# Preview first email
if raw_emails:
    print(f"\nSample email ID: {raw_emails[0]['id']}")
    print(raw_emails[0]["raw_text"][:300])


---
## Section 3 — PII Redaction Module
**Owner: [Name]**

Strips sensitive personal and financial information from raw emails **before**
any AI agent processes them. The redacted email is what all downstream stages see.

**PII types to redact (Indian insurance context):**
- Policy numbers, claim reference numbers
- Aadhaar (12-digit), PAN (ABCDE1234F format), driving licence, vehicle registration
- Diagnosis codes, hospital names, treatment details
- Phone numbers, email addresses, physical addresses
- Nominee names, sum assured amounts


In [ ]:
# ── Section 3: Regex-based PII Redaction ─────────────────────────────────────
import re

# PII pattern registry — extend as needed
PII_PATTERNS = {
    "AADHAAR":      r"\b[2-9]{1}[0-9]{3}\s?[0-9]{4}\s?[0-9]{4}\b",
    "PAN":          r"\b[A-Z]{5}[0-9]{4}[A-Z]{1}\b",
    "PHONE":        r"\b(?:\+91[\-\s]?)?[6-9]\d{9}\b",
    "EMAIL":        r"\b[A-Za-z0-9._%+\-]+@[A-Za-z0-9.\-]+\.[A-Za-z]{2,}\b",
    "VEHICLE_REG":  r"\b[A-Z]{2}[\s\-]?[0-9]{1,2}[\s\-]?[A-Z]{1,2}[\s\-]?[0-9]{4}\b",
    "POLICY_NO":    r"\b[A-Z]{2,4}[\-/]?\d{6,12}\b",
    "CLAIM_REF":    r"\bCLM[\-/]?\d{6,12}\b",
    "PINCODE":      r"\b[1-9][0-9]{5}\b",
    "AMOUNT":       r"\b(?:Rs\.?|INR|₹)\s?[\d,]+(?:\.\d{1,2})?\b",
    # TODO: Add driving licence pattern
    # TODO: Add Aadhaar masked format (XXXX XXXX 1234)
    # TODO: Add diagnosis / ICD code pattern if needed
}

def redact_pii(text: str) -> tuple[str, dict]:
    """
    Apply all PII patterns to the input text.
    Returns:
      - redacted_text: str with placeholders like [AADHAAR], [PAN], etc.
      - redaction_log: dict of { pii_type: count } — values never logged, only counts
    """
    redacted = text
    redaction_log = {}

    # TODO: iterate over PII_PATTERNS
    # TODO: for each pattern, use re.subn() to replace matches with [PII_TYPE]
    # TODO: record count in redaction_log
    # TODO: return redacted text and log

    return redacted, redaction_log


In [ ]:
# ── Section 3: spaCy NER pass (names, locations) ─────────────────────────────
import spacy

nlp = spacy.load("en_core_web_sm")

def redact_named_entities(text: str) -> tuple[str, dict]:
    """
    Use spaCy NER to redact PERSON and GPE (location) entities
    not caught by regex patterns.
    Returns redacted text and entity log (type: count).
    """
    doc = nlp(text)
    redacted = text
    entity_log = {}

    # TODO: iterate doc.ents in reverse order (to preserve char offsets)
    # TODO: replace ent.text with [ent.label_] for PERSON, GPE, ORG entities
    # TODO: record counts in entity_log

    return redacted, entity_log


In [ ]:
# ── Section 3: Full redaction pipeline ───────────────────────────────────────

def full_redact(email: dict) -> dict:
    """
    Run both regex and NER redaction on a raw email.
    Returns enriched email dict with:
      - redacted_text: str
      - redaction_log: dict (pii_type: count, no values)
    """
    raw = email["raw_text"]

    # Step 1 — regex patterns
    after_regex, regex_log = redact_pii(raw)

    # Step 2 — spaCy NER
    after_ner, ner_log = redact_named_entities(after_regex)

    combined_log = {**regex_log, **ner_log}

    return {
        **email,
        "redacted_text": after_ner,
        "redaction_log": combined_log,
    }

# Apply to all loaded emails
redacted_emails = [full_redact(e) for e in raw_emails]

# Preview
if redacted_emails:
    sample = redacted_emails[0]
    print(f"Email ID: {sample['id']}")
    print(f"Redaction log: {sample['redaction_log']}")
    print(f"\nRedacted text preview:\n{sample['redacted_text'][:400]}")


---
## Section 4 — RAG Pipeline: Embeddings & ChromaDB
**Owner: [Name]**

Chunks policy documents, generates embeddings, and stores them in ChromaDB.
Exposes a `retrieve(query, insurance_line, top_k)` function used by the RAG Agent in Section 5.

**Design notes:**
- Each chunk is stored with metadata: `source`, `insurance_line`, `chunk_index`
- Retrieval is filtered by `insurance_line` to avoid cross-product confusion
- ChromaDB is persisted to `vectordb/` on Drive so it survives session restarts


In [ ]:
# ── Section 4: Chunk policy documents ────────────────────────────────────────
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.schema import Document

def chunk_documents(docs: list[dict], chunk_size: int, chunk_overlap: int) -> list[Document]:
    """
    Split loaded policy documents into overlapping chunks.
    Returns list of LangChain Document objects with metadata preserved.
    """
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        separators=["\n\n", "\n", ". ", " ", ""],
    )
    chunks = []

    # TODO: for each doc in docs:
    #   split doc["text"] using splitter.split_text()
    #   for each chunk, create Document(page_content=chunk, metadata={
    #       "source": doc["source"],
    #       "insurance_line": doc["insurance_line"],
    #       "chunk_index": i
    #   })

    print(f"Total chunks created: {len(chunks)}")
    return chunks

chunks = chunk_documents(all_docs, CONFIG["chunk_size"], CONFIG["chunk_overlap"])


In [ ]:
# ── Section 4: Build ChromaDB vector store ───────────────────────────────────
import chromadb
from langchain_community.vectorstores import Chroma
from langchain_anthropic import AnthropicEmbeddings
# Alternative: from langchain_community.embeddings import HuggingFaceEmbeddings

def build_vectorstore(chunks: list[Document], persist_dir: str):
    """
    Embed all chunks and persist to ChromaDB.
    Uses Anthropic embeddings by default.
    Swap embedding model here if needed — everything else stays the same.
    """
    # TODO: initialise embedding model
    # embedding_model = AnthropicEmbeddings(model="voyage-3")
    # OR: embedding_model = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

    # TODO: create Chroma vectorstore from documents
    # vectorstore = Chroma.from_documents(
    #     documents=chunks,
    #     embedding=embedding_model,
    #     persist_directory=persist_dir,
    # )
    # vectorstore.persist()

    # TODO: return vectorstore
    pass

vectorstore = build_vectorstore(chunks, CONFIG["vectordb_dir"])
print("ChromaDB vectorstore built and persisted.")


In [ ]:
# ── Section 4: Load existing vectorstore (use after first build) ──────────────

def load_vectorstore(persist_dir: str):
    """
    Load a previously persisted ChromaDB store.
    Call this instead of build_vectorstore() after the first run.
    """
    # TODO: initialise same embedding model as used in build_vectorstore()
    # TODO: return Chroma(persist_directory=persist_dir, embedding_function=embedding_model)
    pass

# Uncomment after first build:
# vectorstore = load_vectorstore(CONFIG["vectordb_dir"])


In [ ]:
# ── Section 4: Retrieval function ────────────────────────────────────────────

def retrieve(query: str, insurance_line: str, top_k: int = CONFIG["top_k"]) -> list[dict]:
    """
    Retrieve top-k relevant policy chunks for a given query and insurance line.
    Filters by insurance_line metadata to avoid cross-product hallucination.

    Returns list of dicts:
      { "content": str, "source": str, "insurance_line": str, "chunk_index": int }
    """
    # TODO: create retriever with metadata filter:
    # retriever = vectorstore.as_retriever(
    #     search_type="similarity",
    #     search_kwargs={
    #         "k": top_k,
    #         "filter": {"insurance_line": insurance_line}
    #     }
    # )
    # TODO: retriever.get_relevant_documents(query)
    # TODO: format results as list of dicts with content + metadata

    results = []
    return results

# Quick test
# results = retrieve("What is the waiting period for pre-existing disease?", "health")
# for r in results:
#     print(r["source"], "->", r["content"][:150])


---
## Section 5 — LangChain Agents: Triage, RAG Retrieval, Resolution
**Owner: [Name]**

Defines three core LangChain agents as a sequential pipeline:
1. **Triage Agent** — classifies the email (insurance line, query type, urgency, risk flags, SLA)
2. **RAG Retrieval Agent** — fetches relevant policy clauses based on triage output
3. **Resolution Agent** — drafts a grounded, cited response or raises escalation flag

All agents use Claude via `langchain-anthropic`. Outputs are structured Pydantic objects
so downstream stages (Section 6, 7, 8) can consume them cleanly.


In [ ]:
# ── Section 5: LLM initialisation ────────────────────────────────────────────
from langchain_anthropic import ChatAnthropic

llm = ChatAnthropic(
    model=CONFIG["llm_model"],
    temperature=CONFIG["llm_temperature"],
    anthropic_api_key=os.environ["ANTHROPIC_API_KEY"],
)

print(f"LLM ready: {CONFIG['llm_model']}")


In [ ]:
# ── Section 5: Shared output schemas (Pydantic) ──────────────────────────────
from pydantic import BaseModel, Field
from typing import Literal, Optional
from datetime import datetime

class TriageOutput(BaseModel):
    insurance_line:    Literal["health", "motor", "life", "travel", "home", "unknown"]
    query_type:        str   # e.g. "new_claim", "claim_status", "coverage_clarification"
    urgency:           Literal["critical", "high", "medium", "low"]
    risk_flags:        list[str]   # e.g. ["claim_rejection_dispute", "legal_threat"]
    escalate:          bool
    escalation_reason: Optional[str]
    sla_tag:           str   # e.g. "3_working_days", "30_days"

class RetrievedChunk(BaseModel):
    content:        str
    source:         str
    insurance_line: str
    chunk_index:    int

class ResolutionOutput(BaseModel):
    draft_reply:        str
    retrieved_chunks:   list[RetrievedChunk]
    confidence_score:   float = Field(ge=0.0, le=1.0)
    escalate:           bool
    escalation_reason:  Optional[str]
    citations:          list[str]   # e.g. ["health_policy.pdf, Section 4.2"]

class CasePayload(BaseModel):
    """Full assembled case passed to audit trail and escalation handler"""
    email_id:        str
    redacted_text:   str
    redaction_log:   dict
    triage:          TriageOutput
    resolution:      Optional[ResolutionOutput]
    created_at:      str = Field(default_factory=lambda: datetime.utcnow().isoformat())


In [ ]:
# ── Section 5: Triage Agent ──────────────────────────────────────────────────
from langchain.prompts import ChatPromptTemplate
from langchain.output_parsers import PydanticOutputParser

triage_parser = PydanticOutputParser(pydantic_object=TriageOutput)

TRIAGE_PROMPT = ChatPromptTemplate.from_messages([
    ("system", """You are FERIA's Triage Agent for an Indian insurance company.
Analyze the customer email and return a structured triage classification.

Insurance lines: health, motor, life, travel, home
Query types: new_claim | claim_status | coverage_clarification | premium_dispute |
             renewal | nominee_update | grievance | general_inquiry
Urgency: critical (hospitalisation/accident) | high | medium | low
Risk flags: claim_rejection_dispute | legal_threat | regulatory_grievance |
            repeated_followup | death_claim | none
SLA tags follow IRDAI mandates:
  - Claim acknowledgement: 3_working_days
  - Final claim decision: 30_days
  - Grievance resolution: 15_days
  - General query: 7_days

{format_instructions}
"""),
    ("human", "Customer email (PII already redacted):\n\n{redacted_email}"),
])

def run_triage_agent(redacted_email: str) -> TriageOutput:
    """
    Run the triage agent on a redacted customer email.
    Returns a structured TriageOutput object.
    """
    # TODO: format prompt with redacted_email and triage_parser.get_format_instructions()
    # TODO: invoke llm with formatted prompt
    # TODO: parse response with triage_parser.parse()
    # TODO: return TriageOutput
    pass


In [ ]:
# ── Section 5: RAG Retrieval Agent ───────────────────────────────────────────

RAG_PROMPT = ChatPromptTemplate.from_messages([
    ("system", """You are FERIA's RAG Retrieval Agent.
Given the customer query and the retrieved policy chunks, identify the most
relevant clauses and return them with precise citations.
Format each citation as: [source_filename, Section X.Y, clause description]
"""),
    ("human", """Customer query summary: {query_summary}
Insurance line: {insurance_line}

Retrieved policy chunks:
{chunks}

Return the 3 most relevant clauses with citations."""),
])

def run_rag_agent(triage: TriageOutput, redacted_email: str) -> list[RetrievedChunk]:
    """
    Retrieve relevant policy chunks based on triage output.
    Calls retrieve() from Section 4, then asks LLM to rank/cite top clauses.
    Returns list of RetrievedChunk objects.
    """
    # TODO: call retrieve() with redacted_email as query and triage.insurance_line
    # TODO: format chunks as readable text for the prompt
    # TODO: invoke llm with RAG_PROMPT to get cited, ranked clauses
    # TODO: parse and return list of RetrievedChunk objects
    pass


In [ ]:
# ── Section 5: Resolution Agent ──────────────────────────────────────────────

RESOLUTION_PROMPT = ChatPromptTemplate.from_messages([
    ("system", """You are FERIA's Resolution Agent for an Indian insurance company.
Draft a professional, empathetic, and accurate response to the customer email.

Rules:
- Use plain language appropriate for a distressed customer
- Ground every claim in the retrieved policy clauses
- Cite the specific policy section for each claim (e.g. "As per Section 4.2 of your Health Policy...")
- Include clear next steps where applicable
- If confidence is below 0.70, set escalate=True and explain why
- Never speculate beyond what the policy documents state
- Comply with IRDAI communication guidelines
"""),
    ("human", """Redacted customer email:
{redacted_email}

Triage result:
{triage}

Retrieved policy context:
{chunks}

Draft the response and assess your confidence score."""),
])

resolution_parser = PydanticOutputParser(pydantic_object=ResolutionOutput)

def run_resolution_agent(
    redacted_email: str,
    triage: TriageOutput,
    chunks: list[RetrievedChunk],
) -> ResolutionOutput:
    """
    Draft a policy-grounded email response.
    Auto-escalates if confidence < threshold or triage.escalate is True.
    Returns ResolutionOutput with draft reply, citations, confidence score.
    """
    # TODO: if triage.escalate is already True, skip drafting and return
    #       ResolutionOutput with escalate=True and triage.escalation_reason

    # TODO: format chunks into readable text for the prompt
    # TODO: invoke llm with RESOLUTION_PROMPT
    # TODO: parse with resolution_parser
    # TODO: if confidence_score < CONFIG["auto_resolve_confidence_threshold"],
    #       force escalate=True with reason "low_confidence"
    # TODO: return ResolutionOutput
    pass


In [ ]:
# ── Section 5: Full agent pipeline ───────────────────────────────────────────

def run_feria_pipeline(email: dict) -> CasePayload:
    """
    End-to-end pipeline for a single redacted email.
    Runs: Triage → RAG Retrieval → Resolution
    Returns a CasePayload ready for audit logging and UI rendering.
    """
    print(f"Processing email: {email['id']}")

    # Stage 1 — Triage
    triage = run_triage_agent(email["redacted_text"])
    print(f"  Triage: {triage.insurance_line} | {triage.query_type} | {triage.urgency}")

    # Stage 2 — RAG Retrieval
    chunks = run_rag_agent(triage, email["redacted_text"])
    print(f"  Retrieved {len(chunks)} policy chunks")

    # Stage 3 — Resolution (skipped if triage forces escalation)
    resolution = run_resolution_agent(email["redacted_text"], triage, chunks)
    print(f"  Escalate: {resolution.escalate} | Confidence: {resolution.confidence_score:.2f}")

    return CasePayload(
        email_id=email["id"],
        redacted_text=email["redacted_text"],
        redaction_log=email["redaction_log"],
        triage=triage,
        resolution=resolution,
    )

# Test on first email
# if redacted_emails:
#     case = run_feria_pipeline(redacted_emails[0])
#     print(case.model_dump_json(indent=2))


---
## Section 6 — Escalation Handler Bot
**Owner: [Name]**

For escalated cases, opens a LangChain conversational chain pre-loaded with full case context.
A human support agent can ask questions and get policy-grounded answers with citations.
The bot explicitly defers all final decisions to the human agent.

**Key differentiator of FERIA** — this is human-in-the-loop AI, not full automation.


In [ ]:
# ── Section 6: Escalation context builder ────────────────────────────────────

def build_escalation_context(case: CasePayload) -> str:
    """
    Build a structured context block pre-loaded into the escalation bot.
    This is what the bot 'knows' before the human agent asks their first question.
    """
    # TODO: format case into a readable context string covering:
    #   - Email ID and creation timestamp
    #   - Redacted email text
    #   - Triage: insurance_line, query_type, urgency, risk_flags, escalation_reason
    #   - Retrieved policy chunks with citations
    #   - Draft response attempted (if any) and confidence score
    #   - Applicable IRDAI SLA tag
    context = ""
    return context


In [ ]:
# ── Section 6: Escalation bot chain ──────────────────────────────────────────
from langchain.memory import ConversationBufferMemory
from langchain.chains import ConversationChain
from langchain.prompts import PromptTemplate

ESCALATION_SYSTEM = """You are FERIA's Escalation Handler Bot assisting a human insurance support agent.

You have full context of the escalated case loaded below. Use it to answer the agent's questions
accurately, citing specific policy sections where relevant.

Important rules:
- Always cite the policy source (document name, section) for factual claims
- Never make a final decision — defer all final calls to the human agent
- If something is outside the policy documents, say so explicitly
- Flag any regulatory (IRDAI) implications the agent should be aware of

Case Context:
{case_context}

Conversation so far:
{history}

Agent: {input}
Bot:"""

def create_escalation_bot(case: CasePayload) -> ConversationChain:
    """
    Create a conversational escalation bot pre-loaded with case context.
    Returns a LangChain ConversationChain ready for multi-turn dialogue.
    """
    context = build_escalation_context(case)

    prompt = PromptTemplate(
        input_variables=["history", "input"],
        partial_variables={"case_context": context},
        template=ESCALATION_SYSTEM,
    )

    memory = ConversationBufferMemory(human_prefix="Agent", ai_prefix="Bot")

    # TODO: return ConversationChain(llm=llm, prompt=prompt, memory=memory, verbose=False)
    pass

def run_escalation_chat(bot: ConversationChain, agent_message: str) -> str:
    """
    Send one message from the human agent and return the bot's response.
    """
    # TODO: return bot.predict(input=agent_message)
    pass

# Example usage:
# if case.resolution.escalate:
#     bot = create_escalation_bot(case)
#     reply = run_escalation_chat(bot, "What exactly is the customer disputing?")
#     print(reply)


In [ ]:
# ── Section 6: Agent resolution sign-off ─────────────────────────────────────

def log_agent_resolution(case_id: str, resolution_note: str, agent_name: str) -> dict:
    """
    Record the human agent's final resolution decision.
    This closes the escalation loop and feeds into the audit trail (Section 7).
    Returns a resolution record dict.
    """
    # TODO: build and return dict:
    # {
    #   "case_id": case_id,
    #   "agent_name": agent_name,
    #   "resolution_note": resolution_note,
    #   "resolved_at": datetime.utcnow().isoformat(),
    #   "closed_by": "human_agent"
    # }
    pass


---
## Section 7 — Audit Trail & SQLite Logging
**Owner: [Name]**

Every case generates a structured, immutable audit log stored in SQLite.
Supports IRDAI compliance documentation — per-case export, SLA breach flagging,
and full escalation chat history.

**Tables:**
- `cases` — one row per email processed
- `audit_events` — timestamped events per case (triage, retrieval, resolution, escalation)
- `escalation_threads` — chat history for escalated cases


In [ ]:
# ── Section 7: SQLite schema setup ───────────────────────────────────────────
import sqlite3
import json

def init_db(db_path: str):
    """
    Create SQLite tables if they don't exist.
    Call once per session before any logging.
    """
    conn = sqlite3.connect(db_path)
    cur = conn.cursor()

    # cases table — one row per processed email
    cur.execute("""
        CREATE TABLE IF NOT EXISTS cases (
            case_id         TEXT PRIMARY KEY,
            email_id        TEXT NOT NULL,
            created_at      TEXT NOT NULL,
            insurance_line  TEXT,
            query_type      TEXT,
            urgency         TEXT,
            risk_flags      TEXT,   -- JSON array
            escalated       INTEGER,
            confidence      REAL,
            sla_tag         TEXT,
            status          TEXT DEFAULT 'open',  -- open | resolved | escalated
            resolved_at     TEXT,
            agent_name      TEXT,
            resolution_note TEXT
        )
    """)

    # audit_events — timestamped stage-by-stage log
    cur.execute("""
        CREATE TABLE IF NOT EXISTS audit_events (
            id          INTEGER PRIMARY KEY AUTOINCREMENT,
            case_id     TEXT NOT NULL,
            event_type  TEXT NOT NULL,  -- triage | rag_retrieval | resolution | escalation | agent_signoff
            event_data  TEXT,           -- JSON blob
            created_at  TEXT NOT NULL,
            FOREIGN KEY (case_id) REFERENCES cases(case_id)
        )
    """)

    # escalation_threads — multi-turn chat history per escalated case
    cur.execute("""
        CREATE TABLE IF NOT EXISTS escalation_threads (
            id          INTEGER PRIMARY KEY AUTOINCREMENT,
            case_id     TEXT NOT NULL,
            role        TEXT NOT NULL,  -- agent | bot
            message     TEXT NOT NULL,
            created_at  TEXT NOT NULL,
            FOREIGN KEY (case_id) REFERENCES cases(case_id)
        )
    """)

    conn.commit()
    conn.close()
    print(f"Database initialised at: {db_path}")

init_db(CONFIG["db_path"])


In [ ]:
# ── Section 7: Log a completed case ──────────────────────────────────────────

def log_case(case: CasePayload, db_path: str):
    """
    Insert or update a case record in the cases table.
    Also inserts triage and resolution audit events.
    """
    conn = sqlite3.connect(db_path)
    cur = conn.cursor()

    # TODO: insert into cases table from case.model_dump()
    # TODO: insert triage audit event into audit_events
    # TODO: if case.resolution: insert resolution audit event
    # TODO: conn.commit() and conn.close()
    pass

def log_escalation_message(case_id: str, role: str, message: str, db_path: str):
    """
    Append one turn of the escalation chat thread to the database.
    role: 'agent' or 'bot'
    """
    # TODO: insert into escalation_threads
    pass

def log_agent_signoff(case_id: str, resolution_note: str, agent_name: str, db_path: str):
    """
    Record the human agent's final sign-off and close the case.
    """
    # TODO: update cases table: status='resolved', resolved_at, agent_name, resolution_note
    # TODO: insert audit event of type 'agent_signoff'
    pass


In [ ]:
# ── Section 7: Query helpers ──────────────────────────────────────────────────

def get_case(case_id: str, db_path: str) -> dict:
    """Fetch a single case record by ID."""
    # TODO: SELECT * FROM cases WHERE case_id = ?
    pass

def get_open_cases(db_path: str) -> list[dict]:
    """Return all open (unresolved) cases — used by dashboard."""
    # TODO: SELECT * FROM cases WHERE status = 'open' ORDER BY created_at DESC
    pass

def get_escalated_cases(db_path: str) -> list[dict]:
    """Return all escalated cases — used by dashboard escalation queue."""
    # TODO: SELECT * FROM cases WHERE escalated = 1 ORDER BY created_at DESC
    pass

def export_case_audit(case_id: str, db_path: str) -> dict:
    """
    Export the full audit record for a case — for IRDAI compliance documentation.
    Returns dict with: case, audit_events, escalation_thread
    """
    # TODO: join cases + audit_events + escalation_threads for the case_id
    # TODO: return as nested dict suitable for JSON export
    pass


---
## Section 8 — FastAPI Backend + Static HTML Server
**Owner: [Name]**

Exposes the FERIA pipeline as a REST API served from Colab using `nest_asyncio` + `uvicorn`.
Also serves the static HTML frontend from `static/`.

**Routes:**
- `POST /ingest-email` — run full pipeline on a single email
- `GET  /cases` — list all cases (for dashboard)
- `GET  /cases/{case_id}` — get single case detail
- `POST /escalation/{case_id}/chat` — send message to escalation bot
- `POST /escalation/{case_id}/resolve` — agent sign-off
- `GET  /audit/{case_id}/export` — export full audit record


In [ ]:
# ── Section 8: FastAPI app setup ─────────────────────────────────────────────
from fastapi import FastAPI, HTTPException
from fastapi.staticfiles import StaticFiles
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel as PydanticBase

app = FastAPI(
    title="FERIA — Financial Email Resolution & Inquiry Agent",
    description="Agentic AI for insurance customer email support | Nexus AI",
    version=CONFIG["app_version"],
)

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_methods=["*"],
    allow_headers=["*"],
)

# Serve static HTML frontend
app.mount("/static", StaticFiles(directory=CONFIG["static_dir"]), name="static")


In [ ]:
# ── Section 8: Request/Response models for API ───────────────────────────────

class IngestEmailRequest(PydanticBase):
    email_id:   str
    raw_text:   str

class IngestEmailResponse(PydanticBase):
    case_id:          str
    insurance_line:   str
    query_type:       str
    urgency:          str
    escalated:        bool
    draft_reply:      str | None
    confidence_score: float | None
    citations:        list[str]

class EscalationChatRequest(PydanticBase):
    agent_message: str

class EscalationChatResponse(PydanticBase):
    bot_reply: str

class AgentResolveRequest(PydanticBase):
    agent_name:      str
    resolution_note: str


In [ ]:
# ── Section 8: API routes ─────────────────────────────────────────────────────

# In-memory store for active escalation bots (session-scoped)
active_bots: dict[str, object] = {}

@app.post("/ingest-email", response_model=IngestEmailResponse)
async def ingest_email(req: IngestEmailRequest):
    """
    Full pipeline: redact → triage → RAG → resolution → log
    """
    # TODO: call full_redact({ "id": req.email_id, "raw_text": req.raw_text })
    # TODO: call run_feria_pipeline(redacted_email)
    # TODO: call log_case(case, CONFIG["db_path"])
    # TODO: if case.resolution.escalate: create_escalation_bot and store in active_bots
    # TODO: return IngestEmailResponse built from case
    pass

@app.get("/cases")
async def list_cases():
    """Return all cases — open, escalated, resolved"""
    # TODO: return get_open_cases(CONFIG["db_path"]) + get_escalated_cases(...)
    pass

@app.get("/cases/{case_id}")
async def get_case_detail(case_id: str):
    """Return full case detail for the dashboard case view"""
    # TODO: call get_case(case_id, CONFIG["db_path"])
    # TODO: raise HTTPException(404) if not found
    pass

@app.post("/escalation/{case_id}/chat", response_model=EscalationChatResponse)
async def escalation_chat(case_id: str, req: EscalationChatRequest):
    """Send agent message to escalation bot and return bot reply"""
    # TODO: look up active_bots[case_id], raise 404 if not found
    # TODO: call run_escalation_chat(bot, req.agent_message)
    # TODO: log both turns with log_escalation_message()
    # TODO: return EscalationChatResponse(bot_reply=reply)
    pass

@app.post("/escalation/{case_id}/resolve")
async def resolve_case(case_id: str, req: AgentResolveRequest):
    """Human agent signs off and closes the case"""
    # TODO: call log_agent_signoff(case_id, req.resolution_note, req.agent_name, ...)
    # TODO: remove from active_bots if present
    # TODO: return { "status": "resolved", "case_id": case_id }
    pass

@app.get("/audit/{case_id}/export")
async def export_audit(case_id: str):
    """Export full IRDAI-compliant audit record for a case"""
    # TODO: call export_case_audit(case_id, CONFIG["db_path"])
    # TODO: return the audit dict
    pass

@app.get("/health")
async def health():
    return { "status": "ok", "version": CONFIG["app_version"], "model": CONFIG["llm_model"] }


In [ ]:
# ── Section 8: Launch server in Colab ────────────────────────────────────────
import nest_asyncio
import uvicorn
from pyngrok import ngrok

nest_asyncio.apply()

# Expose via ngrok for public URL (useful for team testing)
# !pip install pyngrok -q
# ngrok.set_auth_token("your_ngrok_token")  # get free token at ngrok.com
# public_url = ngrok.connect(8000)
# print(f"FERIA live at: {public_url}")

# Start the server
# uvicorn.run(app, host="0.0.0.0", port=8000)


---
## End-to-End Test
**Run this after all sections are implemented to validate the full pipeline.**


In [ ]:
# ── E2E Test: Single email through full pipeline ──────────────────────────────

SAMPLE_EMAIL = """
Subject: Claim rejection for hospitalization — policy no. AB-123456

Dear Sir/Madam,

I was hospitalized at Apollo Hospital, Mumbai from 12th to 18th June 2025
for a cardiac procedure. My Aadhaar is 9876 5432 1098 and PAN is ABCDE1234F.
My policy number is AB-123456 and claim reference is CLM-789012.

I am deeply distressed to inform you that my claim of Rs. 4,50,000 has been
rejected citing a pre-existing condition clause. I have been a policyholder
for 6 years and was not informed of this exclusion at the time of purchase.

I request an immediate review and escalation to your grievance officer.
I am considering approaching IRDAI if this is not resolved within 3 days.

Regards,
Rajesh Kumar
+91 98765 43210
"""

# Step 1 — Redact
test_email_raw = { "id": "TEST_001", "filename": "test.txt", "raw_text": SAMPLE_EMAIL }
test_email_redacted = full_redact(test_email_raw)
print("Redaction log:", test_email_redacted["redaction_log"])
print("\nRedacted text preview:")
print(test_email_redacted["redacted_text"][:400])


In [ ]:
# Step 2 — Run full pipeline (uncomment once agents are implemented)

# case = run_feria_pipeline(test_email_redacted)
# print("\n── Triage ──")
# print(case.triage.model_dump_json(indent=2))
# print("\n── Resolution ──")
# if case.resolution:
#     print(f"Escalate: {case.resolution.escalate}")
#     print(f"Confidence: {case.resolution.confidence_score}")
#     print(f"Draft reply:\n{case.resolution.draft_reply}")


In [ ]:
# Step 3 — Log to SQLite and verify

# log_case(case, CONFIG["db_path"])
# stored = get_case(case.email_id, CONFIG["db_path"])
# print("Stored case:", stored)


In [ ]:
# Step 4 — Escalation bot (if case was escalated)

# if case.resolution and case.resolution.escalate:
#     bot = create_escalation_bot(case)
#     q1 = run_escalation_chat(bot, "What exactly is the customer disputing?")
#     print("Bot:", q1)
#     q2 = run_escalation_chat(bot, "What does our policy say about pre-existing disease waiting periods?")
#     print("Bot:", q2)
